# 🎬 Python Motion Design — Showcase & Interatividade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seu-usuario/skills-motion-design/blob/main/examples/motion_design_showcase.ipynb)

Este notebook demonstra a criação de **Motion Design de alta performance em Python puro** (Pillow + NumPy + FFmpeg), capaz de renderizar:
1. **Cenas Full-Screen (100% Preenchidas em `.mp4`)** com gradientes e contadores numéricos progressivos.
2. **Overlays Transparentes com Canal Alfa Real (`.webm`)** com osciloscópios harmônicos.
3. **Dossiês Investigativos com Marca-Texto Orgânico (Rough Notation)** com perturbações senoidais.

## ⚙️ 1. Configuração do Ambiente Headless (Colab / Jupyter)

In [ ]:
# Instalação das dependências mínimas do sistema e fontes
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-liberation fonts-dejavu-core
!pip install -q pillow numpy

import subprocess
import base64
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import IPython.display as ipd

print("✅ Ambiente configurado com sucesso!")

## 📺 2. Cena Full-Screen: Infográfico 100% Preenchido (.mp4)

In [ ]:
def render_fullscreen_frame(t: float, dur: float = 4.0, w: int = 1280, h: int = 720) -> Image.Image:
    # 1. Gradiente procedural contínuo em NumPy
    t_vals = np.linspace(0, 1, h)[:, None]
    r = (15 * (1 - t_vals) + 8 * t_vals).astype(np.uint8)
    g = (22 * (1 - t_vals) + 10 * t_vals).astype(np.uint8)
    b = (36 * (1 - t_vals) + 14 * t_vals).astype(np.uint8)
    a = np.full((h, 1), 255, dtype=np.uint8)
    arr = np.repeat(np.concatenate([r, g, b, a], axis=1)[:, None, :], w, axis=1)
    img = Image.fromarray(arr, mode="RGBA")
    draw = ImageDraw.Draw(img, "RGBA")

    # Easing easeOutExpo
    enter_t = min(1.0, t / 0.5)
    ease = 1.0 - pow(1.0 - enter_t, 3)
    p_roll = min(1.0, max(0.0, t / (dur * 0.7)))
    ease_roll = 1.0 if p_roll >= 0.999 else (1.0 - pow(2.0, -10.0 * p_roll))

    card_w = int(w * 0.75 * ease)
    card_h = int(h * 0.55)
    x0 = (w - int(w * 0.75)) // 2
    y0 = int(h * 0.22)

    if card_w > 20:
        draw.rounded_rectangle([x0, y0, x0 + card_w, y0 + card_h], radius=12, fill=(17, 24, 39, 235), outline=(245, 158, 11, int(220 * ease)), width=2)
        curr_val = int(384400 * ease_roll)
        draw.text((x0 + 30, y0 + 30), "TELEMETRIA ORBITAL DE MISSÃO", fill=(245, 158, 11, int(255 * ease)))
        draw.text((x0 + 30, y0 + 80), f"{curr_val:,} KM".replace(",", "."), fill=(248, 250, 252, int(255 * ease)))
        
        bar_w = int((card_w - 60) * ease_roll)
        if bar_w > 4:
            draw.rectangle([x0 + 30, y0 + 160, x0 + 30 + bar_w, y0 + 175], fill=(6, 182, 212, 240))

    return img

# Renderização via pipe stdin para MP4 H.264 em ~1.2s
fps = 30
dur = 4.0
output_mp4 = "fullscreen_explainer.mp4"
cmd = [
    "ffmpeg", "-y", "-f", "rawvideo", "-s", "1280x720", "-pix_fmt", "rgba",
    "-r", str(fps), "-i", "pipe:0", "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-preset", "fast", "-crf", "18", output_mp4
]
proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
for f in range(int(fps * dur)):
    frame = render_fullscreen_frame(f / float(fps), dur)
    proc.stdin.write(frame.tobytes("raw", "RGBA"))
proc.stdin.close()
proc.wait()

print(f"✅ Vídeo de tela cheia gerado: {output_mp4}")
ipd.Video(output_mp4, embed=True, width=640)

## 🎭 3. Overlay Transparente com Canal Alfa Real (.webm) e Validação Xadrez

In [ ]:
def render_transparent_hud(t: float, dur: float = 3.5, w: int = 1280, h: int = 720) -> Image.Image:
    # Buffer totalmente transparente (alpha = 0)
    img = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img, "RGBA")

    enter_t = min(1.0, t / 0.45)
    ease = 1.0 - pow(1.0 - enter_t, 3)

    card_w, card_h = int(w * 0.70), int(h * 0.50)
    x0 = (w - card_w) // 2
    y0 = (h - card_h) // 2

    # Card translúcido com neon glow ciano
    panel_bg = (8, 14, 24, int(210 * ease))
    draw.rounded_rectangle([x0, y0, x0 + card_w, y0 + card_h], radius=10, fill=panel_bg, outline=(6, 182, 212, int(240 * ease)), width=2)
    draw.text((x0 + 24, y0 + 20), "HUD ESPECTRAL // OSCILOSCÓPIO HARMÔNICO", fill=(6, 182, 212, int(255 * ease)))

    # Osciloscópio senoidal multi-harmônico
    osc_w = card_w - 48
    center_y = y0 + 120
    pts = []
    for i in range(70):
        rel_x = i / 69.0
        px = x0 + 24 + int(rel_x * osc_w)
        env = np.sin(rel_x * np.pi)
        w1 = np.sin(rel_x * 12.0 + t * 5.0) * 26.0
        w2 = np.sin(rel_x * 24.0 - t * 7.5) * 12.0
        py = center_y + int((w1 + w2) * env * ease)
        pts.append((px, py))

    for j in range(len(pts) - 1):
        draw.line([pts[j], pts[j + 1]], fill=(6, 182, 212, int(255 * ease)), width=2)

    return img

# Renderização com canal alfa real VP9 yuva420p
output_webm = "transparent_hud.webm"
cmd_webm = [
    "ffmpeg", "-y", "-f", "rawvideo", "-s", "1280x720", "-pix_fmt", "rgba",
    "-r", str(fps), "-i", "pipe:0", "-c:v", "libvpx-vp9", "-pix_fmt", "yuva420p",
    "-auto-alt-ref", "0", "-b:v", "0", "-crf", "28", "-deadline", "realtime", output_webm
]
proc = subprocess.Popen(cmd_webm, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
for f in range(int(fps * 3.5)):
    frame = render_transparent_hud(f / float(fps), 3.5)
    proc.stdin.write(frame.tobytes("raw", "RGBA"))
proc.stdin.close()
proc.wait()

# Exibição com fundo quadriculado xadrez para validar transparência
with open(output_webm, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")

html = f"""
<div style="display: inline-block; padding: 10px; border-radius: 8px; background: repeating-conic-gradient(#333 0% 25%, #181818 0% 50%) 50% / 20px 20px;">
  <video width="640" autoplay loop muted playsinline controls>
    <source src="data:video/webm;base64,{b64}" type="video/webm">
  </video>
</div>
"""
print("✅ Vídeo transparente gerado! Repare no fundo xadrez demonstrando o canal alfa real:")
ipd.display(ipd.HTML(html))

## 📑 4. Dossiê Investigativo: Marca-Texto Orgânico (Rough Notation)

In [ ]:
def render_dossier_highlight(t: float = 1.8, dur: float = 3.0, w: int = 1280, h: int = 720) -> Image.Image:
    img = Image.new("RGBA", (w, h), (14, 16, 20, 255))
    draw = ImageDraw.Draw(img, "RGBA")

    card_w, card_h = int(w * 0.80), int(h * 0.55)
    x0 = (w - card_w) // 2
    y0 = (h - card_h) // 2

    draw.rounded_rectangle([x0, y0, x0 + card_w, y0 + card_h], radius=8, fill=(20, 22, 26, 255), outline=(245, 158, 11, 200), width=1)
    draw.text((x0 + 30, y0 + 30), "DOCUMENTO CONFIDENCIAL // REGISTRO OFICIAL", fill=(245, 158, 11, 255))

    # Carimbo
    draw.rectangle([x0 + card_w - 220, y0 + 24, x0 + card_w - 40, y0 + 56], outline=(239, 68, 68, 220), width=2)
    draw.text((x0 + card_w - 200, y0 + 32), "DESCLASSIFICADO", fill=(239, 68, 68, 220))

    # Grifo com perturbação senoidal
    quote = "EVIDÊNCIA CONCLUSIVA: ANOMALIA ELETROMAGNÉTICA DETECTADA"
    text_y = y0 + 120
    hl_w = int(len(quote) * 16 * min(1.0, t / 1.2))
    steps = max(8, hl_w // 16)
    poly = []
    for i in range(steps + 1):
        poly.append((x0 + 28 + int(i * (hl_w / steps)), text_y + int(np.sin(i * 0.8) * 3.0)))
    for i in range(steps, -1, -1):
        poly.append((x0 + 28 + int(i * (hl_w / steps)), text_y + 40 + int(np.cos(i * 0.7) * 2.5)))

    draw.polygon(poly, fill=(245, 158, 11, 100))
    draw.text((x0 + 30, text_y + 8), quote, fill=(248, 250, 252, 255))
    return img

preview = render_dossier_highlight(t=1.8)
ipd.display(preview)